## Imports

In [2]:
import numpy as np
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, normalize
import time
from scipy.stats import mode
%matplotlib inline

## Data

In [3]:
X = pd.read_csv("kmeans_data/data.csv", header=None).values 
y = pd.read_csv("kmeans_data/label.csv", header=None).values.ravel() 

In [4]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print("unique labels:", np.unique(y))
print("label counts:", Counter(y))

X shape: (10000, 784)
y shape: (10000,)
unique labels: [0 1 2 3 4 5 6 7 8 9]
label counts: Counter({1: 1135, 2: 1032, 7: 1028, 3: 1010, 9: 1009, 4: 982, 0: 980, 8: 974, 6: 958, 5: 892})


## Preprocess

In [5]:
scaler = StandardScaler()
X_std = scaler.fit_transform(X)

In [6]:
X_unit = normalize(X, norm='l2', axis=1)

In [7]:
X_nonneg = np.copy(X)
X_nonneg[X_nonneg < 0] = 0  # simple clip; or add a constant to shift if needed

In [8]:
# K = number of clusters
K = len(np.unique(y))
print("K =", K)

K = 10


## Distance Functions

In [9]:
def euclidean_distances(X, centroids):
    # returns shape (n_samples, K)
    # uses broadcasting
    return np.sqrt(((X[:, None, :] - centroids[None, :, :])**2).sum(axis=2))

def cosine_dissimilarity(X, centroids):
    # assumes rows L2-normalized
    # cosine_similarity = dot product
    sim = np.dot(X, centroids.T)  
    return 1.0 - sim

def generalized_jaccard_dissimilarity(X, centroids):
    # assumes X and centroids are non-negative
    # J = sum(min) / sum(max)
    n, d = X.shape
    k = centroids.shape[0]
    dists = np.zeros((n, k))
    for j in range(k):
        a = X
        b = centroids[j]
        numer = np.minimum(a, b).sum(axis=1)
        denom = np.maximum(a, b).sum(axis=1) + 1e-12
        jacc = numer / denom
        dists[:, j] = 1.0 - jacc
    return dists

## Centroids

In [10]:
def init_centroids(X, K, seed=42):
    rng = np.random.RandomState(seed)
    idx = rng.choice(X.shape[0], size=K, replace=False)
    return X[idx].astype(float)

## Kmeans

In [11]:
def compute_SSE(X, centroids, labels, dissimilarity='euclidean'):
    # compute SSE using the corresponding dists; SSE uses squared Euclidean by default,
    # but for generality we compute sum of squared dissimilarities (consistent across metrics)
    if dissimilarity == 'euclidean':
        dists = euclidean_distances(X, centroids)
    elif dissimilarity == 'cosine':
        dists = cosine_dissimilarity(X, centroids)
    elif dissimilarity == 'jaccard':
        dists = generalized_jaccard_dissimilarity(X, centroids)
    chosen = dists[np.arange(len(labels)), labels]
    return (chosen**2).sum()

In [12]:
def kmeans(X, K, metric='euclidean', max_iters=500, tol=1e-6, seed=0, verbose=False):
    # pick appropriate X preprocessing expectation:
    assert metric in ('euclidean','cosine','jaccard')
    centroids = init_centroids(X, K, seed=seed)
    sse_history = []
    prev_centroids = centroids.copy()
    start = time.time()
    for it in range(1, max_iters+1):
        # compute distances
        if metric == 'euclidean':
            dists = euclidean_distances(X, centroids)
        elif metric == 'cosine':
            dists = cosine_dissimilarity(X, centroids)
        else:
            dists = generalized_jaccard_dissimilarity(X, centroids)
        labels = np.argmin(dists, axis=1)
        # compute SSE
        sse = (dists[np.arange(X.shape[0]), labels]**2).sum()
        sse_history.append(sse)
        # update centroids (mean of members)
        new_centroids = np.zeros_like(centroids)
        for k in range(K):
            members = X[labels == k]
            if len(members) == 0:
                # reinitialize empty cluster (random data point)
                new_centroids[k] = X[np.random.randint(0, X.shape[0])]
            else:
                new_centroids[k] = members.mean(axis=0)
        # stopping checks:
        centroid_shift = np.linalg.norm(new_centroids - centroids)
        if verbose: print(f"iter {it}: SSE={sse:.4f}, shift={centroid_shift:.6f}")
        # 1) no change in centroid position
        if centroid_shift <= tol:
            reason = "centroid_no_change"
            centroids = new_centroids
            break
        # 2) SSE increases compared to previous iteration
        if it > 1 and sse_history[-1] > sse_history[-2] + 1e-12:
            reason = "sse_increase"
            centroids = new_centroids
            break
        centroids = new_centroids
    else:
        reason = "max_iter"
    end = time.time()
    # final labels and sse recompute
    if metric == 'euclidean':
        dists = euclidean_distances(X, centroids)
    elif metric == 'cosine':
        dists = cosine_dissimilarity(X, centroids)
    else:
        dists = generalized_jaccard_dissimilarity(X, centroids)
    labels = np.argmin(dists, axis=1)
    final_sse = (dists[np.arange(X.shape[0]), labels]**2).sum()
    return dict(centroids=centroids, labels=labels, sse_history=sse_history,
                iterations=len(sse_history), time_taken=end-start, final_sse=final_sse, stop_reason=reason)

## Cluster Labelling

In [13]:
def cluster_majority_labels(labels, y_true, K):
    cluster_labels = np.zeros(K, dtype=int)
    for k in range(K):
        members = y_true[labels == k]
        if len(members) == 0:
            cluster_labels[k] = -1  # no members
        else:
            m = mode(members, keepdims=True)  # ensures result is array
            cluster_labels[k] = m.mode[0]
    return cluster_labels

In [14]:
def clustering_accuracy(labels, y_true, cluster_labels):
    pred = np.array([cluster_labels[l] if cluster_labels[l] != -1 else -1 for l in labels])
    valid = pred != -1
    acc = (pred[valid] == y_true[valid]).sum() / len(y_true[valid])
    return acc

## Results

In [15]:
results = {}

# Euclidean
res_euc = kmeans(X_std, K, metric='euclidean', max_iters=500, seed=0, verbose=False)
clabels = cluster_majority_labels(res_euc['labels'], y, K)
acc_euc = clustering_accuracy(res_euc['labels'], y, clabels)
res_euc.update({'accuracy': acc_euc})
results['euclidean'] = res_euc

# Cosine (use X_unit)
res_cos = kmeans(X_unit, K, metric='cosine', max_iters=500, seed=0, verbose=False)
clabels = cluster_majority_labels(res_cos['labels'], y, K)
acc_cos = clustering_accuracy(res_cos['labels'], y, clabels)
res_cos.update({'accuracy': acc_cos})
results['cosine'] = res_cos

# Jaccard (use X_nonneg)
res_jac = kmeans(X_nonneg, K, metric='jaccard', max_iters=500, seed=0, verbose=False)
clabels = cluster_majority_labels(res_jac['labels'], y, K)
acc_jac = clustering_accuracy(res_jac['labels'], y, clabels)
res_jac.update({'accuracy': acc_jac})
results['jaccard'] = res_jac

# summary
for name, r in results.items():
    print(name, "final_sse:", r['final_sse'], "accuracy:", r['accuracy'], "iters:", r['iterations'], "time(s):", r['time_taken'], "stop:", r['stop_reason'])

euclidean final_sse: 5555336.185024954 accuracy: 0.5224 iters: 111 time(s): 35.75205087661743 stop: centroid_no_change
cosine final_sse: 2110.4535549873153 accuracy: 0.4735 iters: 2 time(s): 0.10844779014587402 stop: sse_increase
jaccard final_sse: 3686.353147591522 accuracy: 0.5477 iters: 12 time(s): 5.739811897277832 stop: sse_increase


In [16]:
def kmeans_forced_stop(X, K, metric='euclidean', stop_type='centroid', max_iters=500, tol=1e-6, seed=0):
    centroids = init_centroids(X, K, seed)
    sse_history = []
    for it in range(1, max_iters+1):
        if metric == 'euclidean':
            dists = euclidean_distances(X, centroids)
        elif metric == 'cosine':
            dists = cosine_dissimilarity(X, centroids)
        else:
            dists = generalized_jaccard_dissimilarity(X, centroids)
        labels = np.argmin(dists, axis=1)
        new_centroids = np.zeros_like(centroids)
        for k in range(K):
            members = X[labels == k]
            new_centroids[k] = members.mean(axis=0) if len(members) else centroids[k]
        sse = (dists[np.arange(len(labels)), labels]**2).sum()
        sse_history.append(sse)
        shift = np.linalg.norm(new_centroids - centroids)

        # different stopping conditions
        if stop_type == 'centroid' and shift <= tol:
            break
        elif stop_type == 'sse_increase' and it > 1 and sse > sse_history[-2]:
            break
        elif stop_type == 'max_iter' and it == max_iters:
            break
        centroids = new_centroids
    return sse_history[-1], len(sse_history)


In [25]:
for metric in ['euclidean', 'cosine', 'jaccard']:
    for stop in ['centroid', 'sse_increase', 'max_iter']:
        sse, iters = kmeans_forced_stop(X_std if metric=='euclidean' else 
                                        X_unit if metric=='cosine' else X_nonneg,
                                        K, metric=metric, stop_type=stop)
        print(metric, stop, "final SSE:", sse, "iterations:", iters)


euclidean centroid final SSE: 5555336.185024954 iterations: 111
euclidean sse_increase final SSE: 5555336.185024954 iterations: 500
euclidean max_iter final SSE: 5555336.185024954 iterations: 500
cosine centroid final SSE: 1932.7169069690065 iterations: 43
cosine sse_increase final SSE: 2245.085466461329 iterations: 2
cosine max_iter final SSE: 1932.7169069690065 iterations: 500
jaccard centroid final SSE: 3690.8226221009477 iterations: 55
jaccard sse_increase final SSE: 3686.07575672872 iterations: 12
jaccard max_iter final SSE: 3690.8226221009477 iterations: 500


## Checks

In [24]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix

# Standardize again for fair comparison
# X_scaled = StandardScaler().fit_transform(X)

# Fit sklearn KMeans
kmeans_sklearn = KMeans(n_clusters=K, n_init=10, random_state=42)
labels_sklearn = kmeans_sklearn.fit_predict(X)

# Compare SSE
print("Sklearn inertia (SSE):", kmeans_sklearn.inertia_)

# Optional: compare your labels vs sklearn labels (clusters may differ in order)
print("Your final SSE:", res_euc['final_sse'])


Sklearn inertia (SSE): 25320407927.6064
Your final SSE: 5555336.185024954


In [23]:
from scipy.stats import mode

def accuracy_from_labels(true_labels, cluster_labels):
    labels_mapped = np.zeros_like(cluster_labels)
    for i in np.unique(cluster_labels):
        mask = cluster_labels == i
        labels_mapped[mask] = mode(true_labels[mask], keepdims=True).mode[0]
    return np.mean(labels_mapped == true_labels)

acc_sklearn = accuracy_from_labels(y, labels_sklearn)
print("Sklearn Euclidean K-Means accuracy:", acc_sklearn)

Sklearn Euclidean K-Means accuracy: 0.5216


In [20]:
from sklearn.metrics import pairwise_distances

# check a few random samples
i, j = 0, 1
print("Your Jaccard:", 1 - np.minimum(X_nonneg[i], X_nonneg[j]).sum() / np.maximum(X_nonneg[i], X_nonneg[j]).sum())
print("Sklearn Jaccard:", pairwise_distances(X_nonneg[[i]], X_nonneg[[j]], metric='jaccard')[0][0])


Your Jaccard: 0.9062153163152054
Sklearn Jaccard: 0.8436213991769548


c:\Codes\Python3.10.11\lib\site-packages\sklearn\metrics\pairwise.py:2361: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


In [21]:
from sklearn.metrics import pairwise_distances

# check a few random samples
i, j = 0, 1
print("Your Cosine:", 1 - np.minimum(X_nonneg[i], X_nonneg[j]).sum() / np.maximum(X_nonneg[i], X_nonneg[j]).sum())
print("Sklearn Cosine:", pairwise_distances(X_nonneg[[i]], X_nonneg[[j]], metric='cosine')[0][0])


Your Cosine: 0.9062153163152054
Sklearn Cosine: 0.8064388049395022
